# Bedrock Module · Day 14 — AgentCore: Runtime & Policy

**Phase 1 · Module 2: AWS Bedrock & AgentCore** · ~2.5 hrs

On Day 13 you hand-rolled an agent loop — `FakeBedrockAgent` held the reasoning, the tools, the memory, all in one process. That's fine for learning, wrong for production: who *hosts* it, who says which tools it may touch, how does it hold a live connection to an external tool server between calls? **Amazon Bedrock AgentCore** (GA March 2026) is the managed answer. Today you build the four pieces that turn a loose loop into a governed, hosted agent:

1. **Runtime** — the managed host that runs the agent loop and isolates each session.
2. **Gateway** — a tool-routing layer: one front door, many tool targets (Lambda, OpenAPI, **MCP**).
3. **Policy** — per-agent access boundaries the Gateway enforces *before* any tool runs.
4. **Stateful MCP connection** — a persistent tool-server session that survives across invocations.

> **Keyless & offline.** No AWS account. We wrap Day-13-style Python tools in `Fake*` classes named after the real AgentCore components, with the same call shapes. Each maps to a real API in the closing table — the swap is a client line, not a rewrite.

## 0. Setup — config + the tools from Day 13

Same settings-object habit (Day 13 Python). We reuse two familiar tools plus one that reaches an *external* system — that external one is what will need a stateful MCP connection later.

In [1]:
import re, json, time

class AgentCoreSettings:                    # stand-in for pydantic BaseSettings
    aws_region  = "eu-west-2"
    agent_model = "eu.anthropic.claude-sonnet-4-5-20250929-v1:0"
    agent_id    = "loan-assistant"          # the hosted agent's identity

cfg = AgentCoreSettings()

# tools the agent may call (Day 13: an 'action group'; here targets behind the Gateway)
ACCOUNTS = {"ACC-1001": {"balance": 4200.0}, "ACC-2002": {"balance": 55000.0}}

def get_balance(account_id):                # safe read  -> a Lambda target
    a = ACCOUNTS.get(account_id)
    return {"balance": a["balance"]} if a else {"error": "no such account"}

def transfer_funds(account_id, amount):     # state change -> a Lambda target
    a = ACCOUNTS.get(account_id)
    if not a: return {"error": "no such account"}
    if amount >= 10_000: return {"status": "NEEDS_APPROVAL", "amount": amount}
    a["balance"] -= amount
    return {"status": "ok", "balance": a["balance"]}

print("agent:", cfg.agent_id, "| region:", cfg.aws_region)

agent: loan-assistant | region: eu-west-2


## 1. Runtime — a **managed host** for the agent loop

AgentCore **Runtime** is the process that runs your agent: it receives an invocation, drives the reason→act→observe loop, isolates each **session** from the others, and returns a completion plus a trace. You supply the agent's config and tools; the Runtime supplies hosting, identity, and session isolation — the things you'd otherwise write yourself.

In [2]:
class AgentCoreRuntime:
    """Managed host: owns identity, per-session isolation, and the invoke entrypoint."""
    def __init__(self, agent_id, gateway, settings):
        self.agent_id = agent_id
        self.gateway  = gateway               # tools are reached ONLY through here (§2)
        self.cfg      = settings
        self._sessions = {}                   # sessionId -> isolated state

    def invoke(self, text, sessionId):
        sess = self._sessions.setdefault(sessionId, {"last_account": None})
        trace = []
        # --- reason (a real Runtime calls the FM here) ---
        acc = (re.search(r'ACC-\d+', text) or [None])
        acc = acc.group(0) if hasattr(acc, "group") else sess["last_account"]
        if acc: sess["last_account"] = acc
        trace.append({"step": "rationale", "text": f"agent={self.agent_id} sess={sessionId}"})

        low = text.lower()
        if "transfer" in low and acc:
            amt = float(re.search(r'(\d[\d,]*)', low).group(1).replace(",", ""))
            result = self.gateway.route("transfer_funds", {"account_id": acc, "amount": amt}, self.agent_id, trace)
            done = f"transfer → {result}"
        elif "balance" in low and acc:
            result = self.gateway.route("get_balance", {"account_id": acc}, self.agent_id, trace)
            done = f"balance is {result.get('balance', result)}"
        else:
            done = "which account? (session has none yet)"
        return {"completion": done, "trace": trace, "sessionId": sessionId}

print("Runtime class ready — note: tools reached only via self.gateway")

Runtime class ready — note: tools reached only via self.gateway


☝ The Runtime never calls a tool directly — every tool goes through `self.gateway.route(...)`. That indirection is deliberate: it's the seam where **policy** gets enforced and where **many kinds** of tool target plug in. Sessions live in `self._sessions`, isolated by id, just like the managed service.

## 2. Gateway — one front door, many tool **targets**

Real agents call tools of different shapes: an **AWS Lambda**, an **OpenAPI** endpoint, an **MCP server**. **AgentCore Gateway** is the routing layer that hides those differences — the agent asks for a tool by name, the Gateway knows which *target type* backs it and dispatches accordingly. (This is the Strategy pattern from this morning: one `route()` call, a different handler per target.)

In [3]:
class AgentCoreGateway:
    """Routes a tool call to the right target type. Enforces policy first (§3)."""
    def __init__(self, policy=None):
        self.targets = {}        # tool_name -> (target_type, handler)
        self.policy  = policy

    def register(self, name, target_type, handler):
        self.targets[name] = (target_type, handler)

    def route(self, name, args, agent_id, trace):
        if name not in self.targets:
            return {"error": f"no target for {name}"}
        target_type, handler = self.targets[name]
        # --- policy check happens BEFORE the tool runs (filled in §3) ---
        if self.policy and not self.policy.allows(agent_id, name):
            trace.append({"step": "policy", "text": f"DENY {agent_id} -> {name}"})
            return {"error": "access denied by policy", "tool": name}
        trace.append({"step": "invocationInput", "target": target_type, "tool": name, "args": args})
        result = handler(**args)                     # dispatch to the target
        trace.append({"step": "observation", "result": result})
        return result

gw = AgentCoreGateway()
gw.register("get_balance",    "lambda",    get_balance)
gw.register("transfer_funds", "lambda",    transfer_funds)
print("registered targets:", {n: t for n,(t,_) in gw.targets.items()})

registered targets: {'get_balance': 'lambda', 'transfer_funds': 'lambda'}


☝ The Gateway maps each tool name to a **target type** and a handler. Today both are `"lambda"`; in §4 we add an `"mcp"` target with the same `register`/`route` interface. The agent's reasoning code doesn't change when a tool moves from Lambda to MCP — only the registration does. One front door, pluggable backends.

## 3. Policy — per-agent **access boundaries**, enforced at the Gateway

Hosting an agent safely means bounding *what it can do*, independent of what it's asked. **AgentCore policy** attaches an access boundary to an **agent identity**: an allow-list of tools/resources. The Gateway checks it **before** dispatching — so a read-only agent physically cannot call `transfer_funds`, no matter how the prompt is worded. Policy lives outside the model; the model can't argue with it.

In [4]:
class AccessPolicy:
    """agent_id -> set of tools it is allowed to invoke (an access boundary)."""
    def __init__(self, boundaries):
        self.boundaries = boundaries          # {agent_id: {allowed tool names}}

    def allows(self, agent_id, tool):
        return tool in self.boundaries.get(agent_id, set())

policy = AccessPolicy({
    "loan-assistant": {"get_balance", "transfer_funds"},   # full access
    "read-only-bot":  {"get_balance"},                     # boundary: no transfers
})
gw.policy = policy                                         # Gateway now enforces it

# prove the boundary: same tool, two identities
t = []
print("loan-assistant transfer:", gw.route("transfer_funds", {"account_id":"ACC-1001","amount":50}, "loan-assistant", t))
print("read-only-bot  transfer:", gw.route("transfer_funds", {"account_id":"ACC-1001","amount":50}, "read-only-bot",  t))

loan-assistant transfer: {'status': 'ok', 'balance': 4150.0}
read-only-bot  transfer: {'error': 'access denied by policy', 'tool': 'transfer_funds'}


☝ Identical tool call, opposite outcomes — the difference is the **agent identity's boundary**, not the request. `read-only-bot` is denied at the Gateway before `transfer_funds` executes. This is how you give a customer-facing agent a narrow blast radius: it can *look*, it cannot *move money*, and that's guaranteed by infrastructure rather than a prompt instruction.

## 4. Stateful **MCP** connection — a tool server that remembers

Some tools live behind a **Model Context Protocol (MCP)** server — an external process exposing tools *and holding state* (an open case, a session cursor). A fresh connection per call would lose that state. AgentCore keeps a **stateful MCP session** open across invocations and routes MCP tools to it. We add a `FakeMCPServer` with internal state and register it as an `"mcp"` target — same Gateway interface.

In [5]:
class FakeMCPServer:
    """External MCP tool server that keeps state between calls (a persistent session)."""
    def __init__(self):
        self.connected = False
        self.notes = []                       # server-side state that must persist

    def connect(self):                        # AgentCore opens this ONCE, reuses it
        self.connected = True
        return {"mcp": "connected", "sessionState": "fresh"}

    def add_case_note(self, text):
        assert self.connected, "MCP session not open"
        self.notes.append(text)               # accumulates across invocations
        return {"note_count": len(self.notes), "latest": text}

mcp = FakeMCPServer()
print(mcp.connect())                          # stateful session opened once
gw.register("add_case_note", "mcp", mcp.add_case_note)   # MCP tool behind the same Gateway
policy.boundaries["loan-assistant"].add("add_case_note") # allow it for our agent

# two separate invocations — the MCP server REMEMBERS between them
t = []
print(gw.route("add_case_note", {"text": "customer called re: overdraft"}, "loan-assistant", t))
print(gw.route("add_case_note", {"text": "escalated to risk team"},        "loan-assistant", t))
print("server-side notes persisted:", mcp.notes)

{'mcp': 'connected', 'sessionState': 'fresh'}
{'note_count': 1, 'latest': 'customer called re: overdraft'}
{'note_count': 2, 'latest': 'escalated to risk team'}
server-side notes persisted: ['customer called re: overdraft', 'escalated to risk team']


☝ `note_count` climbed 1 → 2 across **two independent Gateway calls** because the MCP session stayed open — the server kept its state instead of resetting. That's the point of a *stateful* connection: the tool server is a long-lived collaborator, not a stateless function. AgentCore manages the lifecycle (connect, reuse, close) so your agent code doesn't.

## 5. End to end — Runtime → Gateway → Policy → targets

Now wire the Runtime on top of the Gateway and drive a multi-turn session. Watch the trace: every tool hop shows its **target type** and passes the **policy** gate before running.

In [6]:
runtime = AgentCoreRuntime(cfg.agent_id, gw, cfg)

def show(resp):
    print("AGENT:", resp["completion"])
    for s in resp["trace"]:
        if s["step"] == "invocationInput": print(f"   ⚙ {s['target']}:{s['tool']}({s['args']})")
        elif s["step"] == "observation":   print(f"   👁 {s['result']}")
        elif s["step"] == "policy":        print(f"   ⛔ {s['text']}")
        else:                              print(f"   💭 {s['text']}")
    print()

SID = "sess-alpha"
show(runtime.invoke("balance on ACC-2002?", SID))
show(runtime.invoke("transfer 500 from it", SID))     # 'it' resolved from session state
show(runtime.invoke("transfer 20,000 from it", SID))  # tool policy: NEEDS_APPROVAL

AGENT: balance is 55000.0
   💭 agent=loan-assistant sess=sess-alpha
   ⚙ lambda:get_balance({'account_id': 'ACC-2002'})
   👁 {'balance': 55000.0}

AGENT: transfer → {'status': 'ok', 'balance': 54500.0}
   💭 agent=loan-assistant sess=sess-alpha
   ⚙ lambda:transfer_funds({'account_id': 'ACC-2002', 'amount': 500.0})
   👁 {'status': 'ok', 'balance': 54500.0}

AGENT: transfer → {'status': 'NEEDS_APPROVAL', 'amount': 20000.0}
   💭 agent=loan-assistant sess=sess-alpha
   ⚙ lambda:transfer_funds({'account_id': 'ACC-2002', 'amount': 20000.0})
   👁 {'status': 'NEEDS_APPROVAL', 'amount': 20000.0}



☝ One managed entrypoint (`runtime.invoke`) produced a full governed run: session state carried `ACC-2002` across turns, each tool went through the Gateway with its target type, policy was checked before dispatch, and the £20k transfer hit the tool's own `NEEDS_APPROVAL` guard. Day 13's tangle of concerns is now **separated layers** — exactly what a production runtime buys you.

## 6. How this maps to real AgentCore

| You built | Real AgentCore |
|---|---|
| `AgentCoreRuntime.invoke` | **AgentCore Runtime** — managed, session-isolated agent host |
| `AgentCoreGateway.route` | **AgentCore Gateway** — tool routing across Lambda / OpenAPI / MCP targets |
| `register(name, "lambda"/"mcp", ...)` | Gateway **targets** (a Lambda, an OpenAPI spec, an MCP server) |
| `AccessPolicy.allows` | **Policy** / access boundaries bound to an agent **identity** |
| `FakeMCPServer` + `connect()` | a **stateful MCP server** connection AgentCore keeps alive |
| `_sessions[sessionId]` | Runtime **session** isolation |

```python
# real shape (needs AWS AgentCore access)
import boto3
core = boto3.client("bedrock-agentcore", region_name="eu-west-2")
core.invoke_agent_runtime(
    agentRuntimeId="loan-assistant",
    sessionId="sess-alpha",
    payload={"inputText": "balance on ACC-2002?"},
)   # Gateway routing + policy enforcement happen inside the managed runtime
```

## 7. Recap — a hosted, governed agent

| Layer | What it gives you |
|---|---|
| **Runtime** | managed host for the agent loop + session isolation |
| **Gateway** | one front door, many tool target types (Lambda/OpenAPI/MCP) |
| **Policy** | per-identity access boundaries, enforced before tools run |
| **Stateful MCP** | persistent tool-server sessions across invocations |

**Barclays lens:** Day 13 proved an agent can *act*; AgentCore is what makes that action **safe to host** — a read-only bot that physically cannot transfer, tools swappable behind a Gateway, and an audit trace of every routed, policy-checked call. Governance is infrastructure here, not prompt etiquette.

